In [ ]:
# !pip install sentence-transformers

In [7]:
# imports
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from anthropic import Anthropic
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

# setup
load_dotenv()
pine_client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# creating serverless index
# pine_client.create_index(
#     name="humanitarian-risk",
#     dimension=384,
#     metric="cosine",
#     spec=ServerlessSpec(cloud="aws", region="us-east-1")
# )


In [8]:
df_foodprice_c = pd.read_csv("../data/food_prices_final.csv")
df_poverty_c = pd.read_csv("../data/poverty_final.csv")


pine_index = pine_client.Index("humanitarian-risk")

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

In [5]:
batch_size = 1000
upsert_batch_size = 100

# --- Ingestion of food price data into Pinecone ---
for i in range(9000, len(df_foodprice_c), batch_size):
    batch = df_foodprice_c.iloc[i: i + batch_size]

    documents = batch['text'].tolist()
    ids = [f"food_{j}" for j in range(i, i + len(batch))]
    metadatas = [
        {
            'admin1': row['admin1'],
            'commodity': row['commodity'],
            'date': str(row['date']),
            'price': float(row['price'])
        }
        for _, row in batch.iterrows()
    ]

    # building one list of vector dicts for each row in batch
    vectors_to_upsert = []
    for doc_id, text, metadata in zip(ids, documents, metadatas):
        embedding = embed_model.encode(text).tolist()
        metadata['text'] = text
        vectors_to_upsert.append({
            'id': doc_id,
            'values': embedding,
            'metadata': metadata
        })

    # chunked upsert to stay within Pinecone's payload limits
    for j in range(0, len(vectors_to_upsert), upsert_batch_size):
        pine_index.upsert(
            vectors=vectors_to_upsert[j:j + upsert_batch_size],
            namespace="food_prices"
        )

    print('done batch: ', i)


# --- Ingestion of poverty data into Pinecone ---
for i in range(0, len(df_poverty_c), batch_size):
    batch = df_poverty_c.iloc[i: i + batch_size]

    documents = batch['text'].tolist()
    ids = [f"poverty_{j}" for j in range(i, i + len(batch))]
    metadatas = [
        {
            'admin1': row['provider_admin1_name'],
            'mpi': float(row['mpi']),
            'population': float(row['headcount_ratio']),
            'date': str(row['reference_period_start'])
        }
        for _, row in batch.iterrows()
    ]

    vectors_to_upsert = []
    for doc_id, text, metadata in zip(ids, documents, metadatas):
        embedding = embed_model.encode(text).tolist()
        metadata['text'] = text
        vectors_to_upsert.append({
            'id': doc_id,
            'values': embedding,
            'metadata': metadata
        })

    for j in range(0, len(vectors_to_upsert), upsert_batch_size):
        pine_index.upsert(
            vectors=vectors_to_upsert[j:j + upsert_batch_size],
            namespace="poverty_mpi"
        )

    print(f"Added batch {i}")

done batch:  9000
done batch:  10000
done batch:  11000
done batch:  12000
done batch:  13000
done batch:  14000
done batch:  15000
done batch:  16000
done batch:  17000
done batch:  18000
done batch:  19000
done batch:  20000
done batch:  21000
done batch:  22000
done batch:  23000
done batch:  24000
done batch:  25000
done batch:  26000
done batch:  27000
done batch:  28000
done batch:  29000
done batch:  30000
done batch:  31000
done batch:  32000
done batch:  33000
done batch:  34000
done batch:  35000
done batch:  36000
done batch:  37000
done batch:  38000
done batch:  39000
done batch:  40000
done batch:  41000
done batch:  42000
done batch:  43000
done batch:  44000
done batch:  45000
done batch:  46000
done batch:  47000
done batch:  48000
done batch:  49000
done batch:  50000
done batch:  51000
done batch:  52000
done batch:  53000
done batch:  54000
done batch:  55000
done batch:  56000
done batch:  57000
done batch:  58000
done batch:  59000
done batch:  60000
done batch:  

In [9]:
pine_index.describe_index_stats()

DescribeIndexStatsResponse(dimension=384, total_vector_count=205525, metric='cosine', namespaces=2)

In [11]:
# loading vectors back from Pinecone to check
def retrieve_context(question):
    query_embedding = embed_model.encode(question).tolist()

    food_results = pine_index.query(
        vector=query_embedding,
        namespace="food_prices",
        top_k=9,
        include_metadata=True
    )

    poverty_results = pine_index.query(
        vector=query_embedding,
        namespace="poverty_mpi",
        top_k=9,
        include_metadata=True
    )
    
    food_texts= [match['metadata']['text'] for match in food_results['matches']]
    poverty_texts = [match['metadata']['text'] for match in poverty_results['matches']]
    all_texts = food_texts + poverty_texts
    context = "\n\n".join(all_texts)
    return context

In [12]:
print(retrieve_context("What is the relationship between food prices and poverty in new delhi?"))

In Delhi, the Retail price of Rice was 24.0 INR per KG in 2011-10-15

In Delhi, the Retail price of Rice was 12.0 INR per KG in 2003-10-15

In Delhi, the Retail price of Rice was 8.0 INR per KG in 1994-01-15

In Delhi, the Retail price of Rice was 12.0 INR per KG in 2003-06-15

In Delhi, the Retail price of Rice was 12.0 INR per KG in 2003-12-15

In Delhi, the Retail price of Rice was 12.0 INR per KG in 2004-02-15

In Delhi, the Retail price of Rice was 24.0 INR per KG in 2011-09-15

In Delhi, the Retail price of Rice was 13.0 INR per KG in 2004-12-15

In Delhi, the Retail price of Rice was 13.0 INR per KG in 2000-06-15

In Delhi, 12.63% of population lives in poverty with an MPI score of 0.0571

In Delhi, 3.35% of population lives in poverty with an MPI score of 0.0134

In Delhi, 3.98% of population lives in poverty with an MPI score of 0.0167

In Gujarat, 14.4% of population lives in poverty with an MPI score of 0.059

In Dadra & Nagar Haveli and Daman & Diu, 14.27% of population liv